# Mahabharata knowledge graph: Phase 4b coreference only

Runs `src/phase4b_coref.py` on a Kaggle GPU and nothing else. Its only input is `build/mentions.parquet`, so the corpus, the parva PDFs and Phases 1 to 3 are not needed here.

Everything after coreference (Phase 4 relations, Phase 2b confidence, Phase 4f, Phases 5 to 8) runs on CPU, locally.

## Before you run

1. **Make one private Kaggle dataset** holding a single file: `build/mentions.parquet` from your local copy of the repo (about 5.7 MB).
   Upload the local file rather than regenerating it. Phase 4 joins coreference to sentences on `(parva, sent_id)`, so `coref.parquet` must be computed from the same `mentions.parquet` that your local Phase 4 reads. Cell 1 prints its SHA-256 so you can check.
   **Keep the dataset and this notebook private.** The `sentence` column is verbatim text of a copyrighted translation.
2. **Add Data**: attach that dataset.
3. **Settings → Accelerator → GPU T4 x1 or P100.** The script is hard-coded to `cuda:0`.
4. **Settings → Internet → On**, for `pip install fastcoref` and the `biu-nlp/lingmess-coref` model download.
5. Set `COREF_LIMIT` in Cell 5. `2000` runs a smoke test in a few minutes. `0` runs the full corpus. For the full run use **Save Version → Save & Run All (Commit)** so it survives a closed browser tab.

## Afterwards, locally

Download `mahabharata_coref_outputs.zip` from the Output tab and unzip it into the repo root. It contains `build/coref.parquet`, `build/coref_report.md` and `build/coref_manifest.json`. Then run the rest of `tools/run_all.sh` from `src/test_possessed_kin.py` onward.

In [ ]:
# Cell 1: find mentions.parquet in the attached data and set up a working copy.
import hashlib, json, os, shutil, sys
from pathlib import Path

INPUT = Path(os.environ.get("MBH_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("MBH_WORK", "/kaggle/working/mahabharata-main"))

hits = sorted(INPUT.rglob("mentions*.parquet"), key=lambda p: len(p.parts))
if not hits:
    raise SystemExit("No mentions.parquet under %s. Attach the dataset holding build/mentions.parquet." % INPUT)
if len(hits) > 1:
    print("note: %d candidates, using %s" % (len(hits), hits[0]))
src_parquet = hits[0]

(WORK / "build").mkdir(parents=True, exist_ok=True)
(WORK / "src").mkdir(parents=True, exist_ok=True)
shutil.copy2(src_parquet, WORK / "build/mentions.parquet")
os.chdir(WORK)   # the script reads and writes build/ relative to the repo root

MENTIONS_SHA = hashlib.sha256((WORK / "build/mentions.parquet").read_bytes()).hexdigest()
print("input:", src_parquet)
print("sha256:", MENTIONS_SHA)
print("compare locally: python -c \"import hashlib;print(hashlib.sha256(open('build/mentions.parquet','rb').read()).hexdigest())\"")

In [ ]:
# Cell 2: preflight. The columns phase4b_coref.py reads, checked before anything is installed.
import pandas as pd
NEED = {"parva", "sent_id", "adhyaya_ordinal", "surface", "label", "sentence"}
df = pd.read_parquet("build/mentions.parquet")
missing = NEED - set(df.columns)
if missing:
    raise SystemExit("mentions.parquet lacks columns %s. Is this the Phase 1 output?" % sorted(missing))
s = df.drop_duplicates(subset=["parva", "sent_id"])
print("mentions %d | sentences %d | chapters %d | characters %d" % (
    len(df), len(s), s.groupby(["parva", "adhyaya_ordinal"], dropna=False).ngroups,
    s.sentence.astype(str).str.len().sum()))
del df, s

In [ ]:
# Cell 3: dependencies. torch, pandas and pyarrow ship with Kaggle.
# transformers is pinned below 5: fastcoref does not load on 5.x (docs/DECISIONS.md D16).
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastcoref", "transformers>=4.57,<5"], check=True)

In [ ]:
# Cell 4: the GPU check. Fail here, not an hour in.
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU, then restart. phase4b_coref.py runs on cuda:0.")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

In [ ]:
%%writefile /kaggle/working/mahabharata-main/src/phase4b_coref.py
"""Phase 4b. Coreference resolution over the Debroy prose.

Added because the Phase 6 entry gate failed and the measured reason was that
relations are rarely inside one clause with both ends named. Of the relation
clauses in this corpus, 360 have two proper names, 722 have one name and a
pronoun, 220 have two pronouns and 1,798 have a multiword description such as
"the son of Drona" in an argument slot. This phase is aimed at those 2,740.

Model: biu-nlp/lingmess-coref, run through fastcoref. LingMess rather than the
distilled f-coref, because f-coref got the smoke-test passage wrong in the way
that matters here. On "Bhishma, Shantanu's son, saw Arjuna advancing. He pierced
him with three arrows", f-coref put Bhishma, He and him in one cluster, which
would assert that Arjuna pierced himself. LingMess separates them correctly and
also links "the Pandava" to Arjuna.

Inference is deterministic, so there is no seed to record. The model identifier
and revision are recorded instead, in build/coref_report.md.

Coreference runs over a document, not a sentence. The document here is the
chapter, which is the citation unit, so a resolution never crosses a citation
boundary and every resolved mention inherits the chapter it was found in. Long
chapters are split into windows and coreference is not carried across a window
boundary; the number of splits is reported.
"""
import collections, json, re, sys
import pandas as pd

from transformers.modeling_utils import PreTrainedModel
# transformers 4.57 refuses to dispatch sdpa attention for Longformer and
# raises rather than falling back. LingMess is a Longformer. Force eager.
PreTrainedModel._check_and_adjust_attn_implementation = lambda self, *a, **k: "eager"
from fastcoref import LingMessCoref

MODEL = "biu-nlp/lingmess-coref"
MAX_CHARS = 7000          # about 1,800 wordpieces, inside the 4,096 window
NOT_A_NAME = {"You", "He", "She", "They", "We", "It", "Thus", "Then", "Having",
              "There", "Here", "This", "That", "These", "Those", "But", "And",
              "His", "Her", "Their", "Your", "Who", "What", "When", "Where",
              "Thou", "Thee", "Thy", "Yet", "Now", "One", "Some", "All", "O"}


def load():
    df = pd.read_parquet("build/mentions.parquet")
    lab = df.groupby(df.surface.str.strip()).label.agg(lambda x: x.value_counts().idxmax())
    persons = {s for s, l in lab.items()
               if l == "PERSON" and re.fullmatch(r"[A-Z][a-z]{2,}", s)
               and s not in NOT_A_NAME}
    sents = df.drop_duplicates(subset=["parva", "sent_id"])[
        ["parva", "sent_id", "adhyaya_ordinal", "sentence"]]
    sents = sents.sort_values(["parva", "sent_id"])
    return persons, sents


def build_documents(sents, stats):
    """One document per chapter, split into windows under the token limit.

    Yields (key, text, index) where index maps a document character offset back
    to (sent_id, offset within that sentence).
    """
    for (parva, adh), g in sents.groupby(["parva", "adhyaya_ordinal"], dropna=False):
        buf, index, pos, part = [], [], 0, 0
        for r in g.itertuples():
            s = str(r.sentence)
            if pos + len(s) > MAX_CHARS and buf:
                yield (parva, adh, part), " ".join(buf), index
                stats["windows"] += 1
                part += 1
                buf, index, pos = [], [], 0
            index.append((pos, pos + len(s), int(r.sent_id)))
            buf.append(s)
            pos += len(s) + 1
        if buf:
            yield (parva, adh, part), " ".join(buf), index
            stats["windows"] += 1
        stats["chapters"] += 1
        if part:
            stats["chapters_split"] += 1


FIRST = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours"}
SECOND = {"you", "your", "yours", "yourself", "thou", "thee", "thy", "thine"}


def person_of(surface):
    """1 or 2 for pronouns of address, 3 otherwise.

    "I" and "you" inside a quoted speech resolve to the speaker and the
    addressee, which is a different problem from third-person anaphora and goes
    wrong in different ways. They are tagged so that downstream extraction can
    weigh them separately or drop them, rather than being silently mixed in.
    """
    w = surface.strip().lower()
    if w in FIRST:
        return 1
    if w in SECOND:
        return 2
    return 3


def locate(index, start, end):
    """Map a document span back to (sent_id, start, end) within its sentence."""
    for a, b, sid in index:
        if start >= a and end <= b:
            return sid, start - a, end - a
    return None                      # the span straddles a sentence boundary


def canonical_of(spans, persons):
    """The name a cluster resolves to, or None.

    The name is the most frequent known person surface appearing as a whole
    token inside any mention of the cluster. A cluster whose every mention is a
    pronoun or an unknown description resolves to nothing and is dropped, rather
    than being guessed at.
    """
    votes = collections.Counter()
    for text in spans:
        for tok in re.findall(r"[A-Za-z]+", text):
            if tok in persons:
                votes[tok] += 1
    if not votes:
        return None
    top, n = votes.most_common(1)[0]
    rivals = [k for k, v in votes.items() if k != top and v >= n]
    if rivals:
        return None       # two names with equal claim; do not choose silently
    return top


def main():
    limit = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    persons, sents = load()
    if limit:
        sents = sents.head(limit)
    stats = collections.Counter()
    docs = list(build_documents(sents, stats))

    model = LingMessCoref(device="cuda:0", enable_progress_bar=False)
    out = []
    B = 64
    for i in range(0, len(docs), B):
        batch = docs[i:i + B]
        preds = model.predict(texts=[d[1] for d in batch])
        for (key, text, index), pred in zip(batch, preds):
            parva, adh, _ = key
            for cluster in pred.get_clusters(as_strings=False):
                strings = [text[a:b] for a, b in cluster]
                name = canonical_of(strings, persons)
                if name is None:
                    stats["cluster_unresolved"] += 1
                    continue
                stats["cluster_resolved"] += 1
                for (a, b), s in zip(cluster, strings):
                    if s == name:
                        stats["mention_is_the_name"] += 1
                        continue          # already extractable without coref
                    loc = locate(index, a, b)
                    if loc is None:
                        stats["mention_straddles_sentences"] += 1
                        continue
                    sid, sa, sb = loc
                    out.append({"parva": parva,
                                "adhyaya_ordinal": adh,
                                "sent_id": sid, "start": sa, "end": sb,
                                "surface": s, "canonical": name,
                                "person": person_of(s),
                                "cluster_size": len(cluster)})
                    stats["mention_resolved"] += 1
        print("docs %d/%d resolved=%d" % (min(i + B, len(docs)), len(docs),
                                          stats["mention_resolved"]), flush=True)

    df = pd.DataFrame(out)
    df.to_parquet("build/coref.parquet")
    report(df, stats, len(docs))
    print("windows=%d mentions_resolved=%d distinct_names=%d"
          % (len(docs), len(df), df.canonical.nunique() if len(df) else 0))


def report(df, stats, ndocs):
    with open("build/coref_report.md", "w", encoding="utf-8") as f:
        f.write("# Phase 4b coreference report\n\nGenerated by `src/phase4b_coref.py`.\n\n")
        f.write("- Model: `%s` via fastcoref, run on GPU with eager attention\n" % MODEL)
        f.write("- Inference is deterministic; no seed applies\n")
        f.write("- Document unit: the chapter, split at %d characters\n\n" % MAX_CHARS)
        f.write("| quantity | count |\n|---|---|\n")
        for k in ("chapters", "chapters_split", "windows", "cluster_resolved",
                  "cluster_unresolved", "mention_is_the_name",
                  "mention_straddles_sentences", "mention_resolved"):
            f.write("| %s | %d |\n" % (k, stats[k]))
        if len(df):
            f.write("\n- Distinct canonical names: %d\n" % df.canonical.nunique())
            f.write("\n## By grammatical person\n\n| person | mentions |\n|---|---|\n")
            for k, v in sorted(df.person.value_counts().items()):
                f.write("| %s | %d |\n"
                        % ({1: "first (I, we)", 2: "second (you)",
                            3: "third and descriptions"}[k], v))
            f.write("\n## Most resolved-to names\n\n| name | mentions |\n|---|---|\n")
            for n, c in df.canonical.value_counts().head(20).items():
                f.write("| %s | %d |\n" % (n, c))
        f.write("\n## What is deliberately dropped\n\n"
                "A cluster resolves only if a known person name appears inside one of\n"
                "its mentions. A cluster of pronouns with no name in it is dropped\n"
                "rather than guessed (%d clusters). A cluster with two names of equal\n"
                "frequency is also dropped, because choosing between them silently is\n"
                "exactly the entity-link guess the hard rules forbid.\n"
                % stats["cluster_unresolved"])


if __name__ == "__main__":
    main()

In [ ]:
# Cell 5: run coreference.
# COREF_LIMIT > 0 runs the first N sentences only: a smoke test, not a result. Do not use its coref.parquet.
import subprocess, time
COREF_LIMIT = 0          # e.g. 2000 for a quick check

cmd = [sys.executable, "src/phase4b_coref.py"] + ([str(COREF_LIMIT)] if COREF_LIMIT else [])
print("====", " ".join(cmd), flush=True)
t = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     env=dict(os.environ, PYTHONUNBUFFERED="1"))
for line in p.stdout:
    print(line, end="", flush=True)
code_ = p.wait()
MINUTES = (time.time() - t) / 60
print("\nexit %d after %.1f minutes" % (code_, MINUTES))
if code_:
    raise SystemExit("phase4b_coref.py failed. Nothing was bundled.")

In [ ]:
# Cell 6: the report, a manifest tying the output to its input, and a zip for the Output tab.
# The zip unpacks into the repo root as build/coref.parquet, build/coref_report.md, build/coref_manifest.json.
import zipfile
from IPython.display import Markdown, display

display(Markdown((WORK / "build/coref_report.md").read_text(encoding="utf-8")))

out = pd.read_parquet("build/coref.parquet")
manifest = {
    "mentions_parquet_sha256": MENTIONS_SHA,
    "coref_parquet_sha256": hashlib.sha256((WORK / "build/coref.parquet").read_bytes()).hexdigest(),
    "coref_rows": len(out),
    "coref_limit": COREF_LIMIT,
    "smoke_test_only": bool(COREF_LIMIT),
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "minutes": round(MINUTES, 1),
}
(WORK / "build/coref_manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
print(json.dumps(manifest, indent=1))

dest = Path("/kaggle/working/mahabharata_coref_outputs.zip")
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ["build/coref.parquet", "build/coref_report.md", "build/coref_manifest.json"]:
        z.write(WORK / rel, rel)
print(dest, "%.1f MB" % (dest.stat().st_size / 1e6))
if COREF_LIMIT:
    print("WARNING: smoke test. Set COREF_LIMIT = 0 and re-run before using this coref.parquet.")